In [ ]:
# pull dataset from hugging face, and save 10000 examples as a valid JSON array
#!/usr/bin/env python3
# url: https://huggingface.co/datasets/RZ412/PokerBench

from datasets import load_dataset
import json

# load and sample
dataset = load_dataset("RZ412/PokerBench")
ds10000 = dataset["train"].shuffle(seed=42).select(range(10000))

# Convert to a list of dicts and write a proper JSON array (not JSON Lines)
records = list(ds10000)

# calculate how many chips in hero's stack

with open("pokerbench_10000.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

# validate (optional): try loading the file back
# with open('pokerbench_10000.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)
#     print(len(data), type(data), data[0])


### Set up Experiment 1: CoT without giving answer vs CoT giving answer

Generate CoT prompts

In [10]:

import json

with open("pokerbench_10000.json","r",encoding="utf-8") as f:
    data = json.load(f)

do_not_explain_str = "Decide on an action based on the strength of your hand on this board, your position, and actions before you. Do not explain your answer.\nYour optimal action is:"
cot_str = """Explain your reasoning step by step. Format your answer in this structure:
1. Stage: <Preflop / Flop / Turn / River>
2. Known info: <board cards, hero hand, stack sizes, position>
3. Opponent range estimate: <brief logic>
4. Pot odds and equity estimate: <numbers or qualitative>
5. Action reasoning: <why fold / call / raise>
6. Final decision: <FOLD / CHECK / CALL / BET X / RAISE X / ALL IN>"""

def replace_instructions(instruction, output=None):
    if not output:
        return instruction.replace(do_not_explain_str, cot_str)
    else:
        optimal_answer_str = f"Your optimal action is: {output}\n\n"
        return instruction.replace(do_not_explain_str, optimal_answer_str + cot_str)

# CoT without answer
cot_no_ans = [replace_instructions(example["instruction"], None) for example in data]
with open("pokerbench_cot_no_answer.json", "w", encoding="utf-8") as f:
    json.dump(cot_no_ans, f, ensure_ascii=False, indent=2)

# CoT with answer
cot_with_ans = [replace_instructions(example["instruction"], example["output"]) for example in data]
with open("pokerbench_cot_with_answer.json", "w", encoding="utf-8") as f:
    json.dump(cot_with_ans, f, ensure_ascii=False, indent=2)

In [11]:
# extract 15 cot examples for testing, put into a txt for copy and paste

number_to_extract = 30
with open("pokerbench_cot_noans_short.txt", "w", encoding="utf-8") as f:
    for example in cot_no_ans[:number_to_extract]:
        f.write(example + "\n----------------------------------------\n")
with open("pokerbench_cot_ans_short.txt", "w", encoding="utf-8") as f:
    for example in cot_with_ans[:number_to_extract]:
        f.write(example + "\n----------------------------------------\n")